![imagenes](logo.png)

### Heurística de imputación (recom_imputer)
- Si hay **muchos atípicos** (outliers_% ≥ 5%) o **fuerte asimetría** (|skew| ≥ 1.0) ⇒ usar **median**.  
- En otro caso ⇒ usar **mean**.

### Heurística de escalado (recom_scaler)
- Si outliers_% ≥ 5% ⇒ usar **RobustScaler**.  
- Si no hay muchos outliers y la distribución es casi simétrica (|skew| ≤ 0.5) ⇒ usar **StandardScaler**.  
- En cualquier otro caso ⇒ usar **MinMaxScaler** (por defecto).


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, MinMaxScaler, StandardScaler
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

# --- Cargar CSV (ajusta la ruta si hace falta) ---
#############################################
#############################################

df = pd.read_csv("datos_preproc_demo.csv")

#############################################
#############################################

df.columns

Index(['edad', 'ingreso_mensual', 'talla_cm', 'peso_kg', 'temperatura_c',
       'frecuencia_cardiaca', 'ciudad', 'tipo_servicio', 'nivel',
       'codigo_serie', 'referencia', 'sensor_defectuoso'],
      dtype='object')

In [2]:
df.head()

,edad,ingreso_mensual,talla_cm,peso_kg,temperatura_c,frecuencia_cardiaca,ciudad,tipo_servicio,nivel,codigo_serie,referencia,sensor_defectuoso
0,2.152350,32.363455,73.0,17.0,1344.359160,NaN,A,azul,bajo,NaN,tagD,74.193076
1,7.527457,NaN,610.0,5.0,1034.626809,0.349578,NaN,verde,medio,X,NaN,NaN
2,3.258677,43.865147,421.0,11.0,1301.186344,1.084957,A,rojo,bajo,Z,tagB,NaN
3,4.481654,76.347980,200.0,6.0,1494.730860,0.343660,A,rojo,alto,Z,tagD,NaN
4,NaN,62.913846,307.0,18.0,856.347865,0.543502,A,NaN,medio,Z,tagC,NaN


In [3]:
df.describe()

,edad,ingreso_mensual,talla_cm,peso_kg,temperatura_c,frecuencia_cardiaca,sensor_defectuoso
count,90.000000,97.000000,101.000000,103.000000,98.000000,94.000000,12.000000
mean,4.957169,49.685381,431.643564,14.631068,1222.520024,0.616925,96.604769
std,1.902442,15.501301,213.890326,5.724030,247.276363,0.325108,21.490507
min,1.094274,3.205871,53.000000,5.000000,551.632955,0.065766,71.239586
25%,3.486597,38.849136,268.000000,9.500000,1071.377085,0.367458,80.743819
50%,4.877406,49.765917,445.000000,15.000000,1231.010536,0.583417,94.518452
75%,6.263607,59.465711,594.000000,19.000000,1370.666147,0.915209,104.850883
max,10.236319,87.069702,799.000000,24.000000,1789.884962,1.188695,137.777892


In [4]:
X_train, X_test = train_test_split(df, test_size=0.25, random_state=0)

In [6]:
###########################
########################### DIAGNÓSTICO DE COLUMNAS
###########################


# 1) Detecta numéricas
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
print("Columnas numéricas:", num_cols)

# 2) Funciones auxiliares
def iqr_outlier_stats(s: pd.Series):
    s = pd.to_numeric(s, errors="coerce").dropna()
    if s.empty:
        return 0, 0.0, np.nan, np.nan, np.nan, np.nan
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lim_inf, lim_sup = q1 - 1.5*iqr, q3 + 1.5*iqr
    n_out = int(((s < lim_inf) | (s > lim_sup)).sum())
    prop_out = n_out / len(s)
    return n_out, prop_out, q1, q3, lim_inf, lim_sup

def bounded_guess(s: pd.Series):
    """Detecta si parece estar acotada en [0,1] o [0,100]."""
    s = pd.to_numeric(s, errors="coerce").dropna()
    if s.empty:
        return None
    mn, mx = float(s.min()), float(s.max())
    if 0.0 <= mn and mx <= 1.0:
        return "[0,1]"
    if 0.0 <= mn and mx <= 100.0:
        return "[0,100]"
    return None

if len(num_cols) == 0:
    print("No hay columnas numéricas en X_train. No se generan imágenes.")
else:
    # --------- Figura 1: HISTOGRAMAS (todos en subplots) ----------
    n = len(num_cols)
    ncols = min(3, n)
    nrows = math.ceil(n / ncols)

    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(5*ncols, 3.8*nrows))
    axes = np.atleast_1d(axes).ravel()

    for i, col in enumerate(num_cols):
        s = pd.to_numeric(X_train[col], errors="coerce").dropna()
        ax = axes[i]
        if s.empty:
            ax.text(0.5, 0.5, "Sin datos", ha="center", va="center")
            ax.set_title(col); ax.set_xlabel(col); ax.set_ylabel("Frecuencia")
            ax.set_xticks([]); ax.set_yticks([])
        else:
            ax.hist(s, bins=30)
            ax.set_title(col)
            ax.set_xlabel(col)
            ax.set_ylabel("Frecuencia")

    # Oculta subplots sobrantes
    for j in range(len(num_cols), len(axes)):
        axes[j].axis("off")

    fig.suptitle("Histogramas de columnas numéricas", y=1.02, fontsize=12)
    fig.tight_layout()
    fig.savefig("histogramas_numericas.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    # --------- Figura 2: BOXPLOTS (todas en un eje) ----------
    series_pairs = []
    for c in num_cols:
        v = pd.to_numeric(X_train[c], errors="coerce").dropna().values
        if v.size > 0:
            series_pairs.append((c, v))

    if len(series_pairs) == 0:
        print("No hay datos numéricos válidos para boxplots. No se genera boxplot.")
        box_path = None
    else:
        labels = [c for c, _ in series_pairs]
        values = [v for _, v in series_pairs]

        fig2 = plt.figure(figsize=(1.6*len(labels)+4, 5))
        plt.boxplot(values, vert=True, showmeans=True)
        plt.xticks(ticks=range(1, len(labels)+1), labels=labels, rotation=35, ha="right")
        plt.ylabel("Valor")
        plt.title("Boxplots de columnas numéricas")
        plt.tight_layout()
        fig2.savefig("boxplots_numericas.png", dpi=150, bbox_inches="tight")
        plt.close(fig2)


# ================================
# Diagnóstico numérico + heurística
# ================================
diagnostico = []
for c in num_cols:
    s = pd.to_numeric(X_train[c], errors="coerce")
    miss_pct = s.isna().mean() * 100
    s_no_na = s.dropna()
    skew = s_no_na.skew() if s_no_na.size > 1 else np.nan
    kurt = s_no_na.kurt() if s_no_na.size > 1 else np.nan
    n_out, prop_out, q1, q3, li, ls = iqr_outlier_stats(s)
    bounds = bounded_guess(s)

    # Heurística de imputación
    if (prop_out >= 0.05) or (pd.notna(skew) and abs(skew) >= 1.0):
        imputador = "median"
    else:
        imputador = "mean"

    # Heurística de escalado
    if prop_out >= 0.05:
        escalador = "RobustScaler"
    elif pd.notna(skew) and abs(skew) <= 0.5:
        escalador = "StandardScaler"
    else:
        escalador = "MinMaxScaler"  # por defecto en casos no normales o acotados

    diagnostico.append({
        "columna": c,
        "missing_%": round(miss_pct, 2),
        "skew": round(skew, 3) if pd.notna(skew) else np.nan,
        "kurtosis": round(kurt, 3) if pd.notna(kurt) else np.nan,
        "outliers_n": n_out,
        "outliers_%": round(prop_out*100, 2),
        "q1": q1, "q3": q3, "IQR": (q3 - q1),
        "lim_inf": li, "lim_sup": ls,
        "bounded": bounds,
        "recom_imputer": imputador,
        "recom_scaler": escalador,
    })

diag_df = pd.DataFrame(diagnostico).sort_values(["outliers_%","missing_%"], ascending=False)
print("\n=== Diagnóstico numérico (heurística) ===\n")

##########################################################
##########################################################

#print(diag_df)

##########################################################
##########################################################

# Sugerencias de bloques numéricos (incluye median+Robust)
suggest_mean_rob = diag_df.query("recom_imputer=='mean' and recom_scaler=='RobustScaler'")["columna"].tolist()
suggest_med_rob  = diag_df.query("recom_imputer=='median' and recom_scaler=='RobustScaler'")["columna"].tolist()
suggest_med_min  = diag_df.query("recom_imputer=='median' and recom_scaler=='MinMaxScaler'")["columna"].tolist()
suggest_med_std  = diag_df.query("recom_imputer=='median' and recom_scaler=='StandardScaler'")["columna"].tolist()
suggest_mean_min = diag_df.query("recom_imputer=='mean' and recom_scaler=='MinMaxScaler'")["columna"].tolist()
suggest_mean_std = diag_df.query("recom_imputer=='mean' and recom_scaler=='StandardScaler'")["columna"].tolist()

print("\nSugerencias de bloques numéricos (auto):")
print("median+Robust   :", suggest_med_rob)
print("median+MinMax   :", suggest_med_min)
print("median+Standard :", suggest_med_std)
print("mean+Robust     :", suggest_mean_rob)
print("mean+MinMax     :", suggest_mean_min)
print("mean+Standard   :", suggest_mean_std)


Columnas numéricas: ['edad', 'ingreso_mensual', 'talla_cm', 'peso_kg', 'temperatura_c', 'frecuencia_cardiaca', 'sensor_defectuoso']

=== Diagnóstico numérico (heurística) ===


Sugerencias de bloques numéricos (auto):
median+Robust   : ['sensor_defectuoso']
median+MinMax   : []
median+Standard : []
mean+Robust     : []
mean+MinMax     : []
mean+Standard   : ['temperatura_c', 'edad', 'frecuencia_cardiaca', 'ingreso_mensual', 'peso_kg', 'talla_cm']


In [15]:
# --- Listas de columnas ---

# Numéricas
# num_med_rob_cols: inputacion mediana con escalado robusto
# num_med_min_cols: inputación mediana con escalado min_max
# num_med_std_cols: inputacion mediana con escalado Estandar
# num_mean_rob_cols: inputación media con escalado robusto
# num_mean_min_cols: inputacion media con escalado min_max
# num_mean_std_cols: inputacion media con escalado Estandar

# Categóricas
# cat_ohe_cols: inputacion moda con OneHot
# cat_ord_cols: inputacion moda con Ordinal

#########################################################
#########################################################

#num_med_rob_cols = []   # mediana + Robust
#num_med_min_cols = []   # mediana + MinMax
#num_med_std_cols = []   # mediana + Estandar
num_mean_rob_cols = ['peso_kg']   # media + Robust
#num_mean_min_cols  = []    # media + MinMax
num_mean_std_cols  = ['temperatura_c', 'edad', 'frecuencia_cardiaca', 'ingreso_mensual', 'talla_cm']    # media + Standard

cat_ohe_cols = ["ciudad","tipo_servicio"]                  # moda + OneHot
cat_ord_cols = ["nivel","codigo_serie"]                  # moda + Ordinal

#########################################################
#########################################################

passthrough_cols = ["referencia"]                      # pasar sin procesar
drop_cols        = ["sensor_defectuoso"]                         # eliminar

###### orden de las categorías ordinales
categorias_ordinales = [
    ["bajo","medio","alto"],
    ["Z","Y","X"]

]

#########################################################
#########################################################

In [16]:
# --- Pipelines NUMÉRICOS ---
'''pipe_med_rob = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  RobustScaler())
])'''

'''pipe_med_min = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  MinMaxScaler())
])'''

'''pipe_med_std = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler",  StandardScaler())
])'''


pipe_mean_rob = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler",  RobustScaler())
])

'''pipe_mean_min = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler",  MinMaxScaler())
])'''

pipe_mean_std = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler",  StandardScaler())
])

# --- Pipelines CATEGÓRICOS ---
pipe_cat_ohe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),  # moda
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

pipe_cat_ord = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),  # moda
    ("encoder", OrdinalEncoder(categories=categorias_ordinales,
                               handle_unknown="use_encoded_value", unknown_value=-1))
])


In [17]:
# --- ColumnTransformer unificado ---
preprocessor = ColumnTransformer(
    transformers=[
        #("num_med_rob", pipe_med_rob, num_med_rob_cols),
        #("num_med_min", pipe_med_min, num_med_min_cols),
        #("num_med_std", pipe_med_std, num_med_std_cols),
        ("num_mean_rob",  pipe_mean_rob,  num_mean_rob_cols),
        #("num_mean_min",  pipe_mean_min,  num_mean_min_cols),
        ("num_mean_std",  pipe_mean_std,  num_mean_std_cols),

        # Categóricos
        ("cat_ohe",      pipe_cat_ohe,  cat_ohe_cols),
        ("cat_ord",      pipe_cat_ord,  cat_ord_cols),

        # Passthrough (sin preprocesar)
        ("passthrough",  "passthrough", passthrough_cols),

        # Drop explícito
        ("drop_high_na", "drop",        drop_cols),
    ],
    remainder="drop",                        # descarta cualquier otra columna no listada
    verbose_feature_names_out=False
)

In [18]:
# ---------- Ajuste y transformación ----------
preprocessor.fit(X_train)

X_train_proc = preprocessor.transform(X_train)
X_test_proc  = preprocessor.transform(X_test)

print("Shape train ->", X_train_proc.shape)
print("Shape test  ->", X_test_proc.shape)


Shape train -> (90, 15)
Shape test  -> (30, 15)


In [19]:
# Reconstruir DataFrame con nombres de columnas
cols_out = preprocessor.get_feature_names_out()
X_train_proc_df = pd.DataFrame(X_train_proc, columns=cols_out)
X_test_proc_df = pd.DataFrame(X_test_proc, columns=cols_out)
X_train_proc_df

,peso_kg,temperatura_c,edad,frecuencia_cardiaca,ingreso_mensual,talla_cm,ciudad_A,ciudad_B,ciudad_C,tipo_servicio_azul,tipo_servicio_rojo,tipo_servicio_verde,nivel,codigo_serie,referencia
0,-0.883761,1.00006,0.503425,0.06481,-0.546558,-1.807589,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,tagC
1,-0.083761,0.0,0.0,-0.189309,-0.0,1.57079,1.0,0.0,0.0,0.0,0.0,1.0,2.0,0.0,tagD
2,-1.150427,0.703157,1.146304,1.841945,1.743394,-0.138569,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,tagD
3,1.249573,0.535489,0.0,2.189994,-0.046168,-1.192422,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,tagA
4,-1.150427,1.184127,-0.176401,-0.880751,2.199625,-1.066363,1.0,0.0,0.0,0.0,1.0,0.0,2.0,0.0,tagD
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,1.116239,0.0,-1.31062,1.183619,-1.235378,-0.113357,1.0,0.0,0.0,1.0,0.0,0.0,1.0,2.0,tagB
86,0.049573,0.566952,-1.926134,0.220914,0.309103,-0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,tagB
87,0.049573,-0.152821,1.190909,0.0,0.565331,-0.456238,1.0,0.0,0.0,0.0,0.0,1.0,2.0,0.0,tagC
88,1.249573,2.348876,1.858364,0.0,-0.116372,0.335413,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,tagA


In [20]:
X_train_proc_df.to_csv("Entrenamiento_procesado.csv",index=False)
X_test_proc_df.to_csv("Prueba_procesado.csv",index=False)

In [21]:
# Columnas escaladas que indicaste:

#############################################
#############################################

cols_escaladas_usuario = ["peso_kg",'temperatura_c', 'edad', 'frecuencia_cardiaca', 'ingreso_mensual', 'talla_cm']

#############################################
#############################################

cols = [c for c in cols_escaladas_usuario if c in X_train_proc_df.columns]
print(f"Columnas escaladas encontradas en X_train_proc_df: {cols}")

if len(cols) == 0:
    print("No hay columnas escaladas válidas en X_train_proc_df. No se generan imágenes.")
else:
    # ---------- Figura 1: HISTOGRAMAS (todas en subplots) ----------
    n = len(cols)
    ncols = min(3, n)
    nrows = math.ceil(n / ncols)

    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(5*ncols, 3.8*nrows))
    axes = np.atleast_1d(axes).ravel()

    for i, col in enumerate(cols):
        s = pd.to_numeric(X_train_proc_df[col], errors="coerce").dropna()
        ax = axes[i]
        if s.empty:
            ax.text(0.5, 0.5, "Sin datos", ha="center", va="center")
            ax.set_title(col); ax.set_xlabel(col); ax.set_ylabel("Frecuencia")
            ax.set_xticks([]); ax.set_yticks([])
        else:
            ax.hist(s, bins=30)
            ax.set_title(col)
            ax.set_xlabel(col)
            ax.set_ylabel("Frecuencia")

    # Ocultar subplots sobrantes
    for j in range(len(cols), len(axes)):
        axes[j].axis("off")

    fig.suptitle("Histogramas (columnas escaladas)", y=1.02, fontsize=12)
    fig.tight_layout()
    hist_file = "scaled_histogramas.png"
    fig.savefig(hist_file, dpi=150, bbox_inches="tight")
    plt.close(fig)

    # ---------- Figura 2: BOXPLOTS (todas en un solo eje) ----------
    series_pairs = []
    for c in cols:
        v = pd.to_numeric(X_train_proc_df[c], errors="coerce").dropna().values
        if v.size > 0:
            series_pairs.append((c, v))

    if len(series_pairs) == 0:
        print("No hay datos válidos para boxplots. No se genera boxplot.")
        box_file = None
    else:
        labels = [c for c, _ in series_pairs]
        values = [v for _, v in series_pairs]

        fig2 = plt.figure(figsize=(1.6*len(labels)+4, 5))
        plt.boxplot(values, vert=True, showmeans=True)
        plt.xticks(ticks=range(1, len(labels)+1), labels=labels, rotation=35, ha="right")
        plt.ylabel("Valor")
        plt.title("Boxplots (columnas escaladas)")
        plt.tight_layout()
        box_file = "scaled_boxplots.png"
        fig2.savefig(box_file, dpi=150, bbox_inches="tight")
        plt.close(fig2)

    print("Imágenes guardadas:")
    print(" -", hist_file)
    if box_file: print(" -", box_file)


Columnas escaladas encontradas en X_train_proc_df: ['peso_kg', 'temperatura_c', 'edad', 'frecuencia_cardiaca', 'ingreso_mensual', 'talla_cm']
Imágenes guardadas:
 - scaled_histogramas.png
 - scaled_boxplots.png
